<a href="https://colab.research.google.com/github/emanhassan2020/HandsOn/blob/main/LLM/florence_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install  transformers==4.49.0 einops timm accelerate sentence-transformers faiss-cpu pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into a

In [5]:
!pip install  transformers

In [2]:
from typing import List, Dict, Any
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForMultimodalLM # AutoModelForMultimodalLM requires transformers >= 4.38.0
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

class ImageRAGSystem:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # 1. Initialize Florence-2 for Captioning
        self.florence_id = "microsoft/Florence-2-large"
        self.caption_processor = AutoProcessor.from_pretrained(self.florence_id, trust_remote_code=True)
        self.caption_model = AutoModelForMultimodalLM.from_pretrained(
            self.florence_id, trust_remote_code=True
        ).to(self.device).eval()

        # 2. Initialize Text Embeddings (T4 Friendly)
        self.embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=self.device)

        # 3. Database Metadata & Index Storage
        self.image_metadata: Dict[int, Dict[str, Any]] = {}
        self.index = None

    @torch.no_grad()
    def generate_caption(self, image: Image.Image, prompt: str = "<MORE_DETAILED_CAPTION>") -> str:
        """Extracts deep visual descriptions from an image."""
        inputs = self.caption_processor(text=prompt, images=image, return_tensors="pt").to(self.device)
        generated_ids = self.caption_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024,
            num_beams=3
        )
        generated_text = self.caption_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return generated_text

    def ingest_images(self, image_paths: List[str]):
        """Processes a collection of images, creates descriptions, and indexes them."""
        captions = []

        for idx, path in enumerate(image_paths):
            try:
                img = Image.open(path).convert("RGB")
                # Generate rich text description
                caption = self.generate_caption(img)
                captions.append(caption)

                # Store metadata reference
                self.image_metadata[idx] = {
                    "path": path,
                    "caption": caption
                }
            except Exception as e:
                print(f"Skipping {path} due to error: {e}")

        if not captions:
            return

        # Generate text embeddings for the captions
        embeddings = self.embed_model.encode(captions, convert_to_numpy=True)
        dimension = embeddings.shape[1]

        # Initialize and populate FAISS Flat L2 Vector Index
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings.astype(np.float32))
        print(f"Successfully indexed {len(captions)} images.")

    def query(self, text_query: str, top_k: int = 2) -> List[Dict[str, Any]]:
        """Retrieves the top-k most visually relevant images matching the query string."""
        if self.index is None:
            raise ValueError("Database is empty. Please run ingest_images() first.")

        # Embed user query
        query_vector = self.embed_model.encode([text_query], convert_to_numpy=True).astype(np.float32)

        # Search vector database
        distances, indices = self.index.search(query_vector, top_k)

        results = []
        for idx in indices[0]:
            if idx in self.image_metadata:
                results.append(self.image_metadata[idx])
        return results

ImportError: cannot import name 'AutoModelForMultimodalLM' from 'transformers' (/usr/local/lib/python3.13/dist-packages/transformers/__init__.py)

In [7]:
import os
start_path = '/content/drive/MyDrive/returning_back/TestData/Images/'

def list_image_files(directory):
    image_files = []
    image_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp')
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(image_extensions):
                image_files.append(os.path.join(root, file))
    return image_files

# Get the list of image files
my_images = list_image_files(start_path)

# Display the list of image files found
print(f"Found {len(my_images)} image files:")
for img_file in my_images:
    print(img_file)

Found 13 image files:
/content/drive/MyDrive/returning_back/TestData/Images/img1.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img2.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img3.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img4.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img5.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img6.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img7.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img8.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img9.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img10.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img11.jpg
/content/drive/MyDrive/returning_back/TestData/Images/img12.jpg
/content/drive/MyDrive/returning_back/TestData/Images/img13.jpg


In [8]:
!pip install transformers==4.49.0

In [9]:
!pip install transformers==4.51.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 30.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.49.0
    Uninstalling transformers-4.49.0:
      Successfully uninstalled transformers-4.49.0


In [8]:
# Initialize system
rag_db = ImageRAGSystem()
start_path = '/content/drive/MyDrive/returning_back/TestData/Images/'
# List your image file locations
my_images = list_image_files(start_path)#["sample_image1.jpg", "sample_image2.png", "office_setup.jpg"]

# Build the vector index
rag_db.ingest_images(image_paths=my_images)

# Query your database
search_results = rag_db.query("baby girl playing with her brother", top_k=1)

for match in search_results:
    print(f"Match Found: {match['path']}")
    print(f"Florence-2 Caption: {match['caption']}\n")


preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py:   0%|          | 0.00/48.7k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] `Florence2Processor` defines `image_processor_class = 'CLIPImageProcessor'`, which is deprecated. Register the correct mapping in `AutoImageProcessor` instead.


config.json:   0%|          | 0.00/2.44k [00:00<?, ?B/s]

configuration_florence2.py:   0%|          | 0.00/15.1k [00:00<?, ?B/s]

AttributeError: 'Florence2LanguageConfig' object has no attribute 'forced_bos_token_id'

In [ ]:
!pip install langgraph langchain-core langchain-openai pydantic

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.language_models import BaseChatModel

# ==========================================
# 1. DEFINE PYDANTIC OUTPUT STRUCTURE
# ==========================================
class RetrievedImageMetadata(BaseModel):
    image_path: str = Field(description="The local or Drive file path to the verified image.")
    confidence_reasoning: str = Field(description="A brief sentence explaining why this image matches the user's intent based on its description.")
    primary_objects: List[str] = Field(description="List of key visual elements or objects found in the image caption.")

class StructuredRAGResponse(BaseModel):
    query: str = Field(description="The initial user search query.")
    image_found: bool = Field(description="True if a closely matching asset was located, False otherwise.")
    matched_image: Optional[RetrievedImageMetadata] = Field(None, description="The metadata for the best match, if found.")

# ==========================================
# 2. DEFINE THE LANGGRAPH STATE
# ==========================================
class RAGState(TypedDict):
    query: str
    retrieved_matches: List[dict]  # Populated by our vector index search
    structured_output: Optional[dict]

# ==========================================
# 3. BUILD THE STRUCTURED NODE PIPELINE
# ==========================================
class LangGraphImageRAG:
    def __init__(self, rag_system_instance, llm: BaseChatModel):
        """
        rag_system_instance: The initialized ImageRAGSystem instance from the previous step.
        llm: A LangChain chat model (e.g., ChatOpenAI, ChatAnthropic, or a local Hugging Face LLM).
        """
        self.rag_db = rag_system_instance
        # Bind the Pydantic schema to force a structured JSON output
        self.structured_llm = llm.with_structured_output(StructuredRAGResponse)

        # Build the graph topology
        builder = StateGraph(RAGState)
        builder.add_node("retrieve_images", self.retrieve_images_node)
        builder.add_node("parse_and_validate", self.parse_and_validate_node)

        builder.add_edge(START, "retrieve_images")
        builder.add_edge("retrieve_images", "parse_and_validate")
        builder.add_edge("parse_and_validate", END)

        self.graph = builder.compile()

    def retrieve_images_node(self, state: RAGState) -> dict:
        """Node 1: Pulls raw matching candidates from the FAISS database index."""
        # Search the database using the prompt query
        matches = self.rag_db.query(state["query"], top_k=2)
        return {"retrieved_matches": matches}

    def parse_and_validate_node(self, state: RAGState) -> dict:
        """Node 2: Evaluates the text descriptions via LLM and builds a Pydantic structure."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", (
                "You are an AI validation agent checking database retrieval matches.\n"
                "Review the user's intent query and the image candidates generated by Florence-2.\n"
                "Select the best candidate and populate the structured schema format accurately."
            )),
            ("human", "User Request: {query}\n\nCandidate Image Details:\n{candidates}")
        ])

        # Format candidates for the LLM
        candidates_str = ""
        for i, match in enumerate(state["retrieved_matches"]):
            candidates_str += f"[{i}] Path: {match['path']}\nCaption: {match['caption']}\n\n"

        # Run inference through structured LLM
        chain = prompt | self.structured_llm
        response: StructuredRAGResponse = chain.invoke({
            "query": state["query"],
            "candidates": candidates_str if candidates_str else "No images found in index."
        })

        # Convert Pydantic object back into standard dictionary for the final graph state
        return {"structured_output": response.model_dump()}


In [ ]:
from langchain_openai import ChatOpenAI

# 1. Initialize your LLM engine
llm_engine = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Initialize the compiled graph wrapper (Passing your existing `rag_db` reference)
pipeline = LangGraphImageRAG(rag_system_instance=rag_db, llm=llm_engine)

# 3. Stream data through the graph pipeline execution
initial_state = {"query": "Find an archive asset showing a work office layout"}
final_output = pipeline.graph.invoke(initial_state)

# 4. View perfectly formatted validated JSON structure
import json
print(json.dumps(final_output["structured_output"], indent=2))


In [ ]:
def query(self, text_query: str, top_k: int = 2) -> List[Dict[str, Any]]:
    """Retrieves the top-k images alongside their FAISS L2 distance scores."""
    if self.index is None:
        raise ValueError("Database is empty. Please run ingest_images() first.")

    query_vector = self.embed_model.encode([text_query], convert_to_numpy=True).astype(np.float32)

    # Capture both distance values and index matrices
    distances, indices = self.index.search(query_vector, top_k)

    results = []
    # distances[0] and indices[0] contain arrays for the first query vector
    for score, idx in zip(distances[0], indices[0]):
        if idx in self.image_metadata:
            match_data = self.image_metadata[idx].copy()
            match_data["distance_score"] = float(score)  # Append score to metadata
            results.append(match_data)
    return results


In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.language_models import BaseChatModel

# Distance Threshold: Values below this are acceptable.
# Adjust this based on your specific embedding model's distribution range.
DISTANCE_THRESHOLD = 0.85

# ==========================================
# 1. SCHEMAS AND STATES
# ==========================================
class RetrievedImageMetadata(BaseModel):
    image_path: str = Field(description="The file path to the verified image.")
    confidence_reasoning: str = Field(description="Explanation of why this image matches.")
    primary_objects: List[str] = Field(description="Key objects found in the description.")

class StructuredRAGResponse(BaseModel):
    query: str = Field(description="The initial search query.")
    image_found: bool = Field(description="True if an asset was located within threshold bounds.")
    matched_image: Optional[RetrievedImageMetadata] = Field(None, description="Best match metadata.")

class RAGState(TypedDict):
    query: str
    retrieved_matches: List[dict]
    structured_output: Optional[dict]

# ==========================================
# 2. ROUTING FUNCTION
# ==========================================
def evaluate_confidence_router(state: RAGState) -> str:
    """
    Evaluates the proximity of retrieved results.
    Returns the next node name destination string.
    """
    matches = state.get("retrieved_matches", [])

    if not matches:
        return "skip_llm"

    # Extract the lowest L2 distance score (the best match)
    best_score = min([m["distance_score"] for m in matches])

    # L2 Distance: Closer to 0 means higher similarity
    if best_score <= DISTANCE_THRESHOLD:
        return "parse_and_validate"  # Proceed to LLM processing
    else:
        return "skip_llm"            # Route away from LLM to save latency/tokens

# ==========================================
# 3. PIPELINE WITH ROUTER
# ==========================================
class LangGraphRoutedRAG:
    def __init__(self, rag_system_instance, llm: BaseChatModel):
        self.rag_db = rag_system_instance
        self.structured_llm = llm.with_structured_output(StructuredRAGResponse)

        builder = StateGraph(RAGState)

        # Define graph nodes
        builder.add_node("retrieve_images", self.retrieve_images_node)
        builder.add_node("parse_and_validate", self.parse_and_validate_node)
        builder.add_node("skip_llm", self.skip_llm_node)

        # Linear initialization entry point
        builder.add_edge(START, "retrieve_images")

        # Add the conditional routing rule following retrieval step
        builder.add_conditional_edges(
            "retrieve_images",
            evaluate_confidence_router,
            {
                "parse_and_validate": "parse_and_validate",
                "skip_llm": "skip_llm"
            }
        )

        # Graph exit points
        builder.add_edge("parse_and_validate", END)
        builder.add_edge("skip_llm", END)

        self.graph = builder.compile()

    def retrieve_images_node(self, state: RAGState) -> dict:
        matches = self.rag_db.query(state["query"], top_k=2)
        return {"retrieved_matches": matches}

    def parse_and_validate_node(self, state: RAGState) -> dict:
        """Executed only if matches are highly relevant."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Build structural fields for the primary validated image match candidate."),
            ("human", "User Request: {query}\n\nCandidates:\n{candidates}")
        ])

        candidates_str = ""
        for i, match in enumerate(state["retrieved_matches"]):
            candidates_str += f"[{i}] Path: {match['path']}\nCaption: {match['caption']}\n\n"

        chain = prompt | self.structured_llm
        response = chain.invoke({"query": state["query"], "candidates": candidates_str})
        return {"structured_output": response.model_dump()}

    def skip_llm_node(self, state: RAGState) -> dict:
        """Fallback node executing flat structure returns when database metrics match poorly."""
        fallback_response = StructuredRAGResponse(
            query=state["query"],
            image_found=False,
            matched_image=None
        )
        return {"structured_output": fallback_response.model_dump()}
